# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

# Day we're updating data
update_date = "04-14-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Get rid of missing dates; they won't be counted anyway
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]

# # Find only >= 2024 to start
# metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2021, 11, 1).strftime("%Y-%m-%d")] # Note that those with only years will default to today

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2021, 11, 1).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 4, 14).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025

8576


In [3]:
# Get list of genotypes

os.chdir(home)

genotypes_df = pd.read_excel("genotype_key.xlsx")

genotypes = list(genotypes_df["Genotype"])

print(genotypes)

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'B1.1', 'B1.2', 'B1.3', 'B2.1', 'B2.2', 'B3.1', 'B3.2', 'B3.3', 'B3.4', 'B3.5', 'B3.6', 'B4.1', 'B5.1', 'Minor01', 'Minor04', 'Minor07', 'Minor08', 'Minor09', 'Minor10', 'Minor11', 'Minor12', 'Minor13', 'Minor14', 'Minor15', 'Minor16', 'Minor17', 'Minor18', 'Minor19', 'Minor24', 'Minor25', 'Minor26', 'Minor27', 'Minor28', 'Minor29', 'Minor30', 'Minor31', 'Minor32', 'Minor33', 'Minor34', 'Minor35', 'Minor36', 'Minor37', 'Minor38', 'Minor39', 'Minor40', 'Minor41', 'Minor42', 'Minor43', 'Minor44', 'Minor45', 'Minor46', 'Minor47', 'Minor48', 'B3.7', 'Minor50', 'Minor51', 'C1.1', 'Minor52', 'Minor53', 'B3.11', 'Minor55', 'Minor56', 'Minor57', 'Minor58', 'B3.10', 'C2.1', 'Minor60', 'Minor61', 'B3.8', 'Minor62', 'Minor63', 'B3.12', 'Minor65', 'Minor66', 'Minor67', 'B3.9', 'Minor70', 'Minor71', 'B3.13', 'Minor73', 'Minor74', 'Minor75', 'Minor76', 'Minor77', 'Minor78', 'Minor79', 'Minor80', 'Minor81', 'Minor82', 'Minor83', 'Minor84', 'C3.1', 'Minor86', 'Mino

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

8163


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,Genotype
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-005-original,SRP503016,H5N1,NaN,SRS21079811,False,NaN,D1.3
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-004-original,SRP503016,H5N1,NaN,SRS21079810,False,NaN,B3.13
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-003-original,SRP503016,H5N1,NaN,SRS21079809,False,NaN,B3.13
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-002-original,SRP503016,H5N1,NaN,SRS21079808,False,NaN,B3.13
7,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009088-001-original,SRP503016,H5N1,NaN,SRS21079806,False,NaN,B3.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8671,SRR33124635,WGS,256.70,71164848,PRJNA1102327,SAMN47941268,Viral,24819306,USDA-NVSL,2025,...,2025-04-14 15:20:53,1,25-006248-005,SRP503016,NaN,"MILK, BULK TANK",SRS24711952,False,NaN,D1.1
8672,SRR33124636,WGS,238.91,29071970,PRJNA1102327,SAMN47941267,Viral,10215386,USDA-NVSL,2025,...,2025-04-14 15:20:58,1,25-006248-004,SRP503016,NaN,"MILK, BULK TANK",SRS24711951,False,NaN,D1.1
8673,SRR33124637,WGS,251.06,97654207,PRJNA1102327,SAMN47941266,Viral,34439987,USDA-NVSL,2025,...,2025-04-14 15:21:25,1,25-006248-002,SRP503016,NaN,"MILK, BULK TANK",SRS24711950,False,NaN,D1.1
8674,SRR33124638,WGS,251.54,56712279,PRJNA1102327,SAMN47941265,Viral,20010970,USDA-NVSL,2025,...,2025-04-14 15:20:40,1,25-006243-006,SRP503016,NaN,"MILK, BULK TANK",SRS24711949,False,NaN,D1.1


In [5]:
# Get specific geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

print(genbank_mapping)

metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

print(genbank_mapping["name_state"])
print(len(metadata_genbank))
display(metadata_genbank) # Maybe there is no state information since 3/18/2025?

                    seg_file  \
0      SRR28752446_HA_cns.fa   
8      SRR28752447_HA_cns.fa   
16     SRR28752448_HA_cns.fa   
24     SRR28752449_HA_cns.fa   
32     SRR28752450_HA_cns.fa   
...                      ...   
38678  SRR32654211_HA_cns.fa   
38686  SRR32654212_HA_cns.fa   
38694  SRR32654213_HA_cns.fa   
38702  SRR32654214_HA_cns.fa   
38710  SRR32654216_HA_cns.fa   

                                            seg_seq_name      sra_run seg  \
0      Consensus_SRR28752446_HA_cns_threshold_0.5_qua...  SRR28752446  HA   
8      Consensus_SRR28752447_HA_cns_threshold_0.5_qua...  SRR28752447  HA   
16     Consensus_SRR28752448_HA_cns_threshold_0.5_qua...  SRR28752448  HA   
24     Consensus_SRR28752449_HA_cns_threshold_0.5_qua...  SRR28752449  HA   
32     Consensus_SRR28752450_HA_cns_threshold_0.5_qua...  SRR28752450  HA   
...                                                  ...          ...  ..   
38678  Consensus_SRR32654211_HA_cns_threshold_0.5_qua...  SRR32654211  HA   

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,Genotype,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,NaN,D1.3,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas
1,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,NaN,B3.13,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas
2,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,NaN,B3.13,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas
3,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,NaN,B3.13,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas
4,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,2024,...,NaN,B3.13,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,SRR28752453,HA,PP752677.1,4,A/cattle/Texas/24-009088-001/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4059,SRR32633088,WGS,145.11,67016467,PRJNA1102327,SAMN47290851,Viral,25388495,USDA-NVSL,2025,...,NaN,B3.13,SRR32633088_HA_cns.fa,Consensus_SRR32633088_HA_cns_threshold_0.5_qua...,SRR32633088,HA,PV456280.1,4,A/cat/OR/25-005913-003-original/2025,OR
4060,SRR32633089,WGS,148.14,140023511,PRJNA1102327,SAMN47290850,Viral,52157519,USDA-NVSL,2025,...,NaN,B3.13,SRR32633089_HA_cns.fa,Consensus_SRR32633089_HA_cns_threshold_0.5_qua...,SRR32633089,HA,PV456272.1,4,A/cat/OR/25-005800-002-original/2025,OR
4061,SRR32633090,WGS,148.49,83727737,PRJNA1102327,SAMN47290849,Viral,31619233,USDA-NVSL,2025,...,NaN,D1.3,SRR32633090_HA_cns.fa,Consensus_SRR32633090_HA_cns_threshold_0.5_qua...,SRR32633090,HA,PV456264.1,4,A/cat/OR/25-005800-001-original/2025,OR
4062,SRR32633093,WGS,148.10,105910536,PRJNA1102327,SAMN47290846,Viral,39757708,USDA-NVSL,2025,...,NaN,B3.2,SRR32633093_HA_cns.fa,Consensus_SRR32633093_HA_cns_threshold_0.5_qua...,SRR32633093,HA,PV457240.1,4,A/cattle/CA/25-005677-001-original/2025,CA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

## Get and save collection date

In [ ]:

# # Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank.csv")

SAMN41019237
SAMN41019236
SAMN41019235
SAMN41019234
SAMN41019231
SAMN41019230
SAMN41019229
SAMN41019228
SAMN41019183
SAMN41019227
SAMN41019226
SAMN41019225
SAMN41019224
SAMN41019222
SAMN41019221
SAMN41019220
SAMN41019219
SAMN41019218
SAMN41019182
SAMN41019217
SAMN41019215
SAMN41019213
SAMN41019212
SAMN41019209
SAMN41019366
SAMN41019365
SAMN41019364
SAMN41019363
SAMN41019362
SAMN41019361
SAMN41019359
SAMN41019358
SAMN41019196
SAMN41019357
SAMN41019355
SAMN41019354
SAMN41019353
SAMN41019352
SAMN41019351
SAMN41019350
SAMN41019349
SAMN41019348
SAMN41019195
SAMN41019347
SAMN41019346
SAMN41019345
SAMN41019344
SAMN41019343
SAMN41019342
SAMN41019341
SAMN41019340
SAMN41019339
SAMN41019338
SAMN41019277
SAMN41019274
SAMN41019273
SAMN41019272
SAMN41019271
SAMN41019270
SAMN41019269
SAMN41019268
SAMN41019187
SAMN41019208
SAMN41019181
Unable to find collection date.
SAMN41019207
SAMN41019206
SAMN41019205
SAMN41019204
SAMN41019202
SAMN41019416
SAMN41019415
SAMN41019414
SAMN41019413
SAMN41019412
SAMN41

In [ ]:
# Upload saved data -- if doing this, make sure the above cell is commented out
os.chdir(temp_files + "saved/")
metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv")
os.chdir(temp_files)

# # Get only updated dates

# unknown_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] == "2024") | (metadata_genbank["Collection_Date_Specific"] == "2025")] # Dates we don't have
# known_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] != "2024") & (metadata_genbank["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# # Get new dates also 
# # new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata_genbank[["Collection_Date_Specific"]])

# display(metadata_genbank)

## Get host type

In [8]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['burrowing owl', 'bald eagle', 'hooded merganser', 'cattle', 'american woodcock', 'guineafowl', 'house-mouse', 'rock pigeon', 'pet food', 'goat', 'black-crowned night-heron', 'flamingo', 'cackling goose', 'barn owl', 'domestic-cat', "cooper's hawk", 'house sparrow', 'cattle milk product', 'red-tailed hawk', 'feline', 'dove', 'fox', 'cago', 'common raven', 'turkey', 'savannah cat', 'quail', 'chicken', 'merganser', 'red-breasted merganser', 'cat', 'eurasian collared dove', 'guinea fowl', 'emu', 'mute swan', 'bobcat', 'owl', 'duck', 'american robin', 'lion', 'pelican', 'crow', 'goose', 'american wigeon', 'great blue heron', 'american crow', 'gadwall', 'canada goose', 'trumpeter swan', 'red-shouldered hawk', 'blackbird', 'domestic cat', 'snowy owl', 'snow goose', 'comon-grackle', 'skunk', 'swan', 'turkey vulture', 'green-winged teal', 'mountain lion', 'great horned owl', 'serval', 'pefa', "geoffroy's cat", 'red fox', 'raccoon', 'lynx', 'peafowl', 'raw pet food', 'vulture', 'hawk', 'wester

In [9]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [10]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

In [11]:
print(metadata_genbank)

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
1     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
2     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
3     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
4     SRR28752453        WGS      144.71   37055919  PRJNA1102327   
...           ...        ...         ...        ...           ...   
4059  SRR32633088        WGS      145.11   67016467  PRJNA1102327   
4060  SRR32633089        WGS      148.14  140023511  PRJNA1102327   
4061  SRR32633090        WGS      148.49   83727737  PRJNA1102327   
4062  SRR32633093        WGS      148.10  105910536  PRJNA1102327   
4063  SRR32633094        WGS      148.09  109655942  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0     SAMN41019237          Viral  28164109   USDA-NVSL            2024  ...   
1     SAMN4

## Make FASTA files

In [12]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [15]:
# Create fasta files 

os.chdir(complete_files + "2021-11-01--2025-04-14/")

for pair in fasta_files.keys():
    output_path = complete_files + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/Cattle/Texas/24-009028-019-original/2024|H5N1|2024-03-20|cattle|A1
>A/Cattle/Texas/24-009775-001-original/2024|H5N1|2024-03-10|cattle|A1
>A/Cattle/New Mexico/24-010512-001-original/2024|H5N1|2024-04-02|cattle|A1
>A/Cattle/South Dakota/24-010354-020-original/2024|H5N1|2024-04-05|cattle|A1
>A/Cattle/Texas/24-009110-019-original/2024|H5N1|2024-03-14|cattle|A1
>A/Chicken/Texas/24-008355-002-original/2024|H5N1|2024-03-16|avian|A1
>A/Cattle/South Dakota/24-010354-001-original/2024|H5N1|2024-04-05|cattle|A1
>A/Cattle/Texas/24-009367-005-original/2024|H5N1|2024-03-25|cattle|A1
>A/CATTLE/New Mexico/24-010192-003/2024|H5N1|2024-04-01|cattle|A1
>A/American Crow/Illinois/24-004479-002-original/2024|H5N1|2024-02-08|avian|A1
>A/DOMESTIC-CAT/NM/24-012046-003/2024|H5N1|2024-04-17|feline|A1
>A/TURKEY/MI/24-012707-001/2024|H5N1|2024-04-29|avian|A1
>A/CATTLE/Idaho/24-012341-022-original/2024|H5N1|2024-04-23|cattle|A1
>A/CATTLE/Colorado/24-012225-014/2024|H5N1|2024-04-23|cattle|A1
>A/CHICKEN/Minnesota/

## De-Duplication

In [3]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/11-01-2021--04-14-2025_all_genotypes_namerica/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [4]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(complete_files + "11-01-2021--04-14-2025_all_genotypes/")

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [5]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [6]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

592
592


In [7]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                gisaid_df = dfs_gisaid[gisaid_key][0]
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                full_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")
                full_dfs[andersen_key].append(full_df)

print(full_dfs)



defaultdict(<class 'list'>, {'A1_HA': [     isolate_partial                                        full_header  \
718       017947-002  >A/eared_grebe/Wyoming/22-017947-002/2022|H5N1...   
719       017947-001  >A/eared_grebe/Wyoming/22-017947-001/2022|H5N1...   
720       030264-001  >A/double_crested_cormorant/Wisconsin/22-03026...   
721       024871-012  >A/gadwall/Tennessee/22-024871-012/2022|H5N1|2...   
722       024871-011  >A/gadwall/Tennessee/22-024871-011/2022|H5N1|2...   
...              ...                                                ...   
1431      007087-002  >A/turkey/Missouri/22-007087-002/2022|H5N1|202...   
1432      006945-001  >A/chicken/Delaware/22-006945-001/2022|H5N1|20...   
1433      006944-002  >A/turkey/Missouri/22-006944-002/2022|H5N1|202...   
1434      006948-001  >A/chicken/Maryland/22-006948-001/2022|H5N1|20...   
1435      006945-002  >A/chicken/Delaware/22-006945-002/2022|H5N1|20...   

                                               sequence  
71

In [8]:
# If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             # full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)

## Create FASTA files combining Andersen and GISAID

In [9]:
# Create FASTA files per segment

combined_files = downloads + "Combined_Files/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA
